# Attention Mechanisms and Transformers — From Scratch to Fine-Tuning

**Author:** Shivani Bokka  
**Datasets:** 20 Newsgroups (binary: rec.sport.hockey vs sci.med) + Toy sequence reversal task  
**Goal:** Build attention and Transformer architectures from scratch, then fine-tune DistilBERT

---

## What Is This Notebook About?

This notebook takes you on a journey from the fundamental attention mechanism all the way to fine-tuning a state-of-the-art pretrained model. We build everything from scratch first — so you understand exactly what's happening inside a Transformer — and then show you how to use Hugging Face to leverage pretrained models in minutes.

---

## Why Transformers?

In 2017, the paper "Attention Is All You Need" (Vaswani et al.) changed NLP forever. The Transformer replaced recurrent networks with pure attention, enabling:

- **Full parallelism** during training (no sequential dependency)
- **Direct long-range connections** (token 1 can directly attend to token 500)
- **Scalability** to massive datasets and model sizes

---

## Topics Covered

| # | Topic | Key Idea |
|---|-------|----------|
| 1 | RNN Limitations | No parallelism; lossy compression of long sequences |
| 2 | Scaled Dot-Product Attention | QK^T/√d_k; soft lookup table |
| 3 | Positional Encoding | Sin/cos embeddings encode position without recurrence |
| 4 | Multi-Head Attention | Run multiple attention functions in parallel |
| 5 | Transformer Encoder | MHA + residuals + layer norm + FFN |
| 6 | Causal Masking | How decoders prevent future information leakage |
| 7 | Encoder-Decoder | Sequence-to-sequence with cross-attention |
| 8 | BERT/GPT/T5 | Understanding pretrained model families |
| 9 | DistilBERT Fine-Tuning | Transfer learning in practice |

---

## Step 1 — Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from collections import Counter
import re
import time

# Hugging Face
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All libraries imported successfully!')

---

## Section 2 — The Problem with RNNs

### Two Fundamental Limitations

**Problem 1: No Parallelism**

In an RNN, to compute `h_t` you must first compute `h_{t-1}`, which requires `h_{t-2}`, and so on. This is a sequential dependency — you **cannot** compute all hidden states simultaneously.

For a sequence of length 512, this means 512 sequential operations. On modern GPUs that can run thousands of parallel operations, this is a massive bottleneck.

**Problem 2: Lossy Long-Range Compression**

An RNN must compress the entire context into a fixed-size hidden state vector. As the sequence grows, early information is progressively overwritten. Even LSTMs with gating struggle beyond ~500 tokens.

Consider the sentence:
> "The keys, which I left on the counter next to the coffee machine that I bought last Tuesday, are missing."

By the time an RNN reaches "are missing", it has processed 20+ tokens since "keys". The subject-verb agreement signal may be diluted.

### How Transformers Solve Both Problems

The Transformer's self-attention allows **every token to directly attend to every other token** — no sequential dependency, no compression bottleneck:

```
RNN:         x1→h1→h2→h3→...→h_T   (sequential; h1 far from h_T)
Transformer: each token sees ALL others in ONE parallel operation
```

---

---

## Section 3 — Scaled Dot-Product Attention from Scratch

### The Intuition: A Soft Lookup Table

Imagine a dictionary where keys aren't exact strings but continuous vectors:
- You have a **query** Q (what you're looking for)
- You have a set of **keys** K (what the stored items are indexed by)
- You have a set of **values** V (the actual stored content)

Attention computes how well the query matches each key, then returns a weighted mix of the values:

```
Attention(Q, K, V) = softmax(QK^T / √d_k) · V
```

**Why divide by √d_k?**  
For large d_k, the dot products Q·K can get very large, pushing softmax into regions where gradients are near zero. Dividing by √d_k keeps the variance of the dot products at ~1, ensuring stable gradients.

### Step by Step

1. Compute raw scores: `scores = QK^T / √d_k`  — shape (seq_len, seq_len)
2. Normalize: `weights = softmax(scores, dim=-1)`  — each row sums to 1
3. Aggregate: `output = weights × V`  — weighted sum of value vectors

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, heads, seq_len, d_k)
    K: (batch, heads, seq_len, d_k)
    V: (batch, heads, seq_len, d_v)
    mask: optional boolean mask (True = keep, False = mask out)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (..., seq, seq)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

# Test on a 5-token sequence with d_model=16
torch.manual_seed(42)
seq_len = 5
d_model = 16

# Random embeddings simulating 5 input tokens
X_demo = torch.randn(1, 1, seq_len, d_model)  # batch=1, heads=1

# For self-attention, Q = K = V = X
output, attn_weights = scaled_dot_product_attention(X_demo, X_demo, X_demo)

print(f'Input shape:   {X_demo.shape}')
print(f'Output shape:  {output.shape}')
print(f'Weights shape: {attn_weights.shape}')
print(f'\nAttention weight matrix (5x5):')
print(attn_weights.squeeze().numpy().round(3))

# Plot attention heatmap
fig, ax = plt.subplots(figsize=(6, 5))
weights_np = attn_weights.squeeze().detach().numpy()
sns.heatmap(weights_np, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=[f'Key {i}' for i in range(seq_len)],
            yticklabels=[f'Query {i}' for i in range(seq_len)],
            ax=ax)
ax.set_title('Self-Attention Weight Matrix (5 Tokens × 5 Tokens)')
ax.set_xlabel('Keys (what we look at)')
ax.set_ylabel('Queries (what we look with)')
plt.tight_layout()
plt.show()

### How to Read This Chart: Self-Attention Weight Matrix

This heatmap shows the **attention weights** for a 5-token sequence. It is a 5×5 matrix where each cell represents how much one token attends to another.

- **Rows (y-axis) = Queries** — each row represents one token asking "which other tokens are relevant to me?"
- **Columns (x-axis) = Keys** — each column represents a token being looked at.
- **Cell value** = the attention weight — how much the query token (row) attends to the key token (column).
- **Each row sums to 1.0** — the weights are a probability distribution over all tokens.
- **Darker blue cells** = higher attention weight (more focus on that key).

**What to look for:**
- The **diagonal** (top-left to bottom-right) often has high values — tokens tend to attend strongly to themselves.
- **Off-diagonal high values** reveal interesting relationships — e.g., a pronoun attending strongly to its referent.
- For random embeddings (as in this demo), attention is roughly uniform — there are no linguistic patterns to exploit.
- After training, these patterns become linguistically meaningful.

> **Key insight:** Self-attention allows EVERY token to directly communicate with EVERY other token in a single operation. This is what makes Transformers so powerful for capturing long-range dependencies.

---

## Section 4 — Positional Encoding

### The Problem: Attention Is Order-Agnostic

Self-attention treats its input as a **set** — it doesn't care about the order of tokens. The phrase "dog bites man" and "man bites dog" would produce identical attention outputs if no positional information is added.

### The Solution: Sinusoidal Positional Encodings

We add a **positional encoding vector** to each token embedding before feeding it to the Transformer:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Where:
- `pos` = position in the sequence (0, 1, 2, ...)
- `i` = dimension index (0 to d_model/2)

**Why sinusoids?**
1. Each position gets a unique encoding (no two positions are identical)
2. The model can generalize to longer sequences than seen during training
3. The encodings have a smooth, predictable structure that the model can learn to interpret
4. `PE(pos + k)` can be expressed as a linear function of `PE(pos)` — relative positions are encoded

Let's generate and visualize the PE matrix.

In [ ]:
def positional_encoding(max_len, d_model):
    """Generate the sinusoidal positional encoding matrix."""
    PE = np.zeros((max_len, d_model))
    positions = np.arange(max_len).reshape(-1, 1)  # (max_len, 1)
    dims = np.arange(0, d_model, 2)  # even dimensions: 0, 2, 4, ...
    div_term = np.power(10000.0, dims / d_model)  # (d_model/2,)
    PE[:, 0::2] = np.sin(positions / div_term)   # even dims: sin
    PE[:, 1::2] = np.cos(positions / div_term)   # odd dims: cos
    return PE

MAX_POSITIONS = 50
D_MODEL = 64
PE = positional_encoding(MAX_POSITIONS, D_MODEL)

print(f'Positional Encoding shape: {PE.shape}  (positions × dimensions)')
print(f'PE[0, :6] (position 0, first 6 dims): {PE[0, :6].round(3)}')
print(f'PE[1, :6] (position 1, first 6 dims): {PE[1, :6].round(3)}')

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(PE, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xlabel('Dimension Index (0 to 63)', fontsize=12)
ax.set_ylabel('Position in Sequence (0 to 49)', fontsize=12)
ax.set_title('Positional Encoding Matrix: 50 Positions × 64 Dimensions', fontsize=13)
plt.tight_layout()
plt.show()

### How to Read This Chart: Positional Encoding Matrix

This heatmap visualizes the positional encoding matrix — the signal added to token embeddings so the Transformer knows where each token sits in the sequence.

- **The y-axis** is the position in the sequence (0 = first token, 49 = last token in this example).
- **The x-axis** is the embedding dimension (0 to 63).
- **Each cell color** shows the PE value at that position/dimension — red = +1, blue = -1, white = 0.
- **Each row** is the unique positional encoding vector that gets added to the token at that position.

**What to look for:**
- **Low-frequency waves** (left side of the x-axis, high-index dimensions): these change slowly across positions — they encode coarse position information ("near the beginning" vs "near the end").
- **High-frequency waves** (right side, low-index dimensions): these change rapidly — they encode fine-grained position information ("position 5 vs position 6").
- **Vertical stripes of alternating red/blue** at the left are the fast sin/cos waves.
- **Smoother gradients** at the right are the slow waves.

> **Analogy:** Think of it like a binary clock: the rightmost digit flips every second (fast), while the leftmost digit flips every hour (slow). Together, they uniquely encode any time.

---

## Section 5 — Multi-Head Attention

### Why Multiple Heads?

A single attention function can only learn one type of relationship between tokens. **Multi-head attention** runs the attention operation **h times in parallel**, each with different learned projections:

```
head_i = Attention(Q·W_i^Q, K·W_i^K, V·W_i^V)
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) · W^O
```

Each head learns to look for **different types of relationships**:
- Head 1: syntactic dependencies (subject-verb agreement)
- Head 2: coreference (pronouns and their referents)
- Head 3: positional proximity
- Head 4: semantic similarity
- ... and so on

**Parameter efficiency:** Instead of a single attention over d_model=512 dimensions, we use h=8 heads each over d_k=64 dimensions. Total parameters are the same, but expressivity is much richer.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """(batch, seq, d_model) → (batch, heads, seq, d_k)"""
        batch, seq, _ = x.shape
        x = x.view(batch, seq, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        Q = self.split_heads(self.W_Q(query))
        K = self.split_heads(self.W_K(key))
        V = self.split_heads(self.W_V(value))
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        # Recombine heads
        batch, heads, seq, d_k = attn_out.shape
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch, seq, self.d_model)
        return self.W_O(attn_out), attn_weights

# Demonstrate: 8 heads produce 8 different attention patterns
torch.manual_seed(99)
D_MODEL_DEMO = 64
NUM_HEADS = 8
SEQ_LEN_DEMO = 10

mha = MultiHeadAttention(D_MODEL_DEMO, NUM_HEADS)
X_mha = torch.randn(1, SEQ_LEN_DEMO, D_MODEL_DEMO)

with torch.no_grad():
    out_mha, attn_w_mha = mha(X_mha, X_mha, X_mha)

# attn_w_mha: (1, 8, 10, 10) → 8 attention matrices
print(f'MHA output shape:  {out_mha.shape}')
print(f'Attention weights: {attn_w_mha.shape}  (batch, heads, seq, seq)')

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for head_idx in range(NUM_HEADS):
    ax = axes[head_idx // 4][head_idx % 4]
    w = attn_w_mha[0, head_idx].detach().numpy()
    sns.heatmap(w, ax=ax, cmap='Blues', vmin=0, vmax=0.3,
                xticklabels=False, yticklabels=False, cbar=False)
    ax.set_title(f'Head {head_idx + 1}', fontsize=10)
    ax.set_aspect('equal')

plt.suptitle('8 Attention Heads Produce 8 Different Attention Patterns (10×10 sequence)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### How to Read This Chart: Multi-Head Attention Patterns

This 2×4 grid shows the attention weight matrix for each of the 8 heads in a Multi-Head Attention layer on a 10-token sequence.

- **Each subplot** is one attention head — its own 10×10 attention matrix.
- **Rows = queries, Columns = keys** — darker blue means more attention from that query to that key.
- **Each row sums to 1.0** across all columns (it's a probability distribution).

**What to look for:**
- Even on random input, different heads focus on different patterns — some show strong diagonal patterns (attending to nearby tokens), others show scattered attention.
- After training, each head specializes. In BERT, researchers have found heads that:
  - Attend to the [CLS] token (global information gathering)
  - Follow syntactic dependencies (subject → verb)
  - Handle coreference (pronoun → noun)
  - Focus on positional patterns (attending to the token 2 positions ahead)

> **The diversity of patterns is key**: multiple heads allow the model to simultaneously capture multiple types of linguistic relationships, which a single attention head cannot do.

---

## Section 6 — Transformer Encoder Block

### Architecture of a Single Encoder Layer

```
Input (x)
    │
    ├──────────────────┐
    ↓                  │
 LayerNorm(x)          │  (pre-norm variant)
    │                  │
 MultiHeadAttention    │
    │                  │
    + ─────────────────┘  (residual connection)
    │
    ├──────────────────┐
    ↓                  │
 LayerNorm             │
    │                  │
 FeedForward           │   (two linear layers with ReLU/GELU)
    │                  │
    + ─────────────────┘  (residual connection)
    │
 Output
```

**Why Residual Connections?**  
They allow gradients to flow directly from the output back to early layers, enabling training of very deep networks (12-96 layers in BERT/GPT).

**Why Layer Normalization?**  
Normalizes activations across the feature dimension (not the batch dimension), stabilizing training for variable-length sequences.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # MHA sub-layer with residual
        attn_out, attn_w = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + self.drop(attn_out)
        # FFN sub-layer with residual
        x = x + self.drop(self.ff(self.norm2(x)))
        return x, attn_w


class TransformerEncoder(nn.Module):
    """Full Transformer Encoder: embedding + positional encoding + N blocks + classifier."""
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4,
                 max_len=100, num_classes=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        # Learnable positional encoding
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
        self.last_attn_weights = None

    def forward(self, x):
        batch, seq = x.shape
        positions = torch.arange(seq, device=x.device).unsqueeze(0)  # (1, seq)
        out = self.drop(self.embed(x) + self.pos_embed(positions))
        all_attn = []
        for layer in self.layers:
            out, attn_w = layer(out)
            all_attn.append(attn_w)
        self.last_attn_weights = all_attn  # save for visualization
        out = self.norm(out)
        # Use CLS-like: mean pooling over sequence
        pooled = out.mean(dim=1)
        return self.classifier(pooled)

print('TransformerEncoder defined.')
demo_enc = TransformerEncoder(vocab_size=100, d_model=64, num_heads=4, num_layers=4)
demo_x = torch.randint(1, 100, (2, 20))
demo_out = demo_enc(demo_x)
print(f'Demo output shape: {demo_out.shape}  (2 examples, 2 classes)')

In [ ]:
# Prepare 20newsgroups data (reuse from notebook 3 pattern)
categories = ['rec.sport.hockey', 'sci.med']
newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)
texts_all = newsgroups.data
labels_all = newsgroups.target

VOCAB_SIZE = 5000
MAX_LEN = 100
PAD_IDX = 0

def simple_tokenize(text):
    return re.findall(r'[a-z]+', text.lower())

counter = Counter()
for text in texts_all:
    counter.update(simple_tokenize(text))

most_common = [word for word, _ in counter.most_common(VOCAB_SIZE - 2)]
word2idx = {word: idx+2 for idx, word in enumerate(most_common)}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1

def encode(text, word2idx, max_len):
    tokens = simple_tokenize(text)
    indices = [word2idx.get(t, 1) for t in tokens]
    if len(indices) >= max_len:
        return indices[:max_len]
    else:
        return indices + [0] * (max_len - len(indices))

X_encoded = np.array([encode(t, word2idx, MAX_LEN) for t in texts_all], dtype=np.int64)
y_labels = np.array(labels_all, dtype=np.int64)

X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_encoded, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64
train_dataset = TextDataset(X_train_raw, y_train_raw)
val_dataset = TextDataset(X_val_raw, y_val_raw)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Train Transformer Encoder for 5 epochs
torch.manual_seed(42)
TRANSFORMER_D = 64
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 4
EPOCHS = 5

transformer_model = TransformerEncoder(
    vocab_size=VOCAB_SIZE,
    d_model=TRANSFORMER_D,
    num_heads=TRANSFORMER_HEADS,
    num_layers=TRANSFORMER_LAYERS,
    max_len=MAX_LEN,
    num_classes=2
).to(device)

optimizer = optim.Adam(transformer_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_losses, val_accs = [], []
for epoch in range(EPOCHS):
    transformer_model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = transformer_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(transformer_model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    transformer_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = transformer_model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    acc = correct / total
    val_accs.append(acc)
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}')

scratch_transformer_acc = val_accs[-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, EPOCHS+1), train_losses, marker='o', color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train Loss')
ax1.set_title('Transformer Training Loss')
ax2.plot(range(1, EPOCHS+1), val_accs, marker='o', color='steelblue')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Transformer Validation Accuracy')
ax2.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

---

## Section 7 — Visualizing Trained Transformer Attention

Let's extract attention weights from the trained encoder and visualize all 4 heads across all 4 layers for one example sentence.

In [ ]:
# Extract attention weights from trained encoder — 2×4 grid of 8 heads for one example
transformer_model.eval()
idx2word = {v: k for k, v in word2idx.items()}

sample_text_idx = 0
sample_x = torch.tensor(X_val_raw[[sample_text_idx]], dtype=torch.long).to(device)

with torch.no_grad():
    _ = transformer_model(sample_x)
    all_attn = transformer_model.last_attn_weights  # list of 4 tensors (1, heads, seq, seq)

# Show layer 0 and layer 2, all 4 heads each → 2 rows × 4 cols
tokens = [idx2word.get(int(sample_x[0][j].item()), '<UNK>') for j in range(MAX_LEN)]
display_tokens = tokens[:15]  # Show first 15 tokens for readability
N_SHOW = len(display_tokens)

fig, axes = plt.subplots(2, TRANSFORMER_HEADS, figsize=(18, 8))

for row_idx, layer_idx in enumerate([0, 2]):
    attn_layer = all_attn[layer_idx][0]  # (heads, seq, seq)
    for head_idx in range(TRANSFORMER_HEADS):
        ax = axes[row_idx][head_idx]
        w = attn_layer[head_idx, :N_SHOW, :N_SHOW].cpu().numpy()
        sns.heatmap(w, ax=ax, cmap='Blues', vmin=0, vmax=w.max(),
                    xticklabels=display_tokens, yticklabels=display_tokens,
                    cbar=False)
        ax.set_title(f'Layer {layer_idx+1}, Head {head_idx+1}', fontsize=9)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.suptitle('Transformer Encoder Attention: Layers 1 & 3, All 4 Heads\n(First 15 tokens of a validation example)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### How to Read This Chart: Transformer Attention Heatmaps

This 2×4 grid shows attention patterns for 2 selected layers (Layer 1 and Layer 3) across all 4 heads, for the first 15 tokens of one validation example.

- **Each subplot** = one head in one layer. It shows a 15×15 attention matrix.
- **Rows = queries** (the token asking): which other tokens does this token attend to?
- **Columns = keys** (the token being attended to).
- **Darker blue** = stronger attention.
- **Token labels** on axes show the actual words.

**What to look for:**
- **Layer 1 heads** often show local patterns — attending to nearby words (short-range context).
- **Later layer heads** tend to show more semantic patterns — attending across longer distances.
- If a content word (e.g., "hockey", "player") gets high attention across many queries, it's a high-information word the model considers globally important.
- Heads that look nearly identical are "redundant" — the model doesn't need them. Pruning those heads usually has minimal accuracy impact.

> **Interesting finding from BERT research:** Some heads learn to attend to punctuation, which acts as a "separator" between linguistic segments.

---

## Section 8 — Causal Masking

### Why Mask?

The encoder uses **full attention** — every token can attend to every other token. This is perfect for tasks like classification where we have the full sequence available.

But for **language generation** (like GPT), we must predict token t+1 given only tokens 1...t. If we allowed attention to "see into the future", the model would just copy the answer — no learning would happen.

**Causal masking** prevents this by setting all upper-triangular attention weights to -infinity (which becomes 0 after softmax):

```
Masked scores:                    Attention weights after softmax:
[[s11, -∞,  -∞, -∞, -∞],        [[1.0, 0.0, 0.0, 0.0, 0.0],
 [s21, s22, -∞, -∞, -∞],         [w21, w22, 0.0, 0.0, 0.0],
 [s31, s32, s33, -∞, -∞],   →    [w31, w32, w33, 0.0, 0.0],
 [s41, s42, s43, s44, -∞],        [w41, w42, w43, w44, 0.0],
 [s51, s52, s53, s54, s55]]       [w51, w52, w53, w54, w55]]
```

Token i can only attend to tokens j ≤ i (present and past, not future).

In [ ]:
# Demonstrate causal masking: attention WITHOUT mask vs WITH mask
torch.manual_seed(5)
demo_seq = 6
demo_d = 32
demo_heads = 2

mha_demo = MultiHeadAttention(demo_d, demo_heads)
X_causal = torch.randn(1, demo_seq, demo_d)

# WITHOUT mask — full attention
with torch.no_grad():
    _, w_full = mha_demo(X_causal, X_causal, X_causal)

# WITH causal mask — lower triangular
causal_mask = torch.tril(torch.ones(1, 1, demo_seq, demo_seq)).bool()
with torch.no_grad():
    _, w_masked = mha_demo(X_causal, X_causal, X_causal, mask=causal_mask)

# Use head 0 for visualization
w_full_np = w_full[0, 0].numpy()
w_masked_np = w_masked[0, 0].numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(w_full_np, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=0.5,
            xticklabels=[f't{i}' for i in range(demo_seq)],
            yticklabels=[f't{i}' for i in range(demo_seq)], ax=ax1)
ax1.set_title('Attention WITHOUT Causal Mask\n(full access to all tokens)')
ax1.set_xlabel('Keys (attended to)')
ax1.set_ylabel('Queries (attending from)')

sns.heatmap(w_masked_np, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=0.5,
            xticklabels=[f't{i}' for i in range(demo_seq)],
            yticklabels=[f't{i}' for i in range(demo_seq)], ax=ax2)
ax2.set_title('Attention WITH Causal Mask\n(only past tokens visible)')
ax2.set_xlabel('Keys (attended to)')
ax2.set_ylabel('Queries (attending from)')

plt.suptitle('Effect of Causal Masking on Attention Weights', fontsize=12)
plt.tight_layout()
plt.show()

### How to Read This Chart: Causal Masking

Two side-by-side 6×6 attention weight heatmaps showing the same attention head, with and without a causal mask.

- **Left (no mask):** All 36 cells have non-zero values — every token can attend to every other token, including future tokens. The row sums are 1.0 across all 6 tokens.
- **Right (causal mask):** The upper triangle is all zeros (shown as white cells). Token t0 can only see t0. Token t3 can see t0, t1, t2, t3 — but not t4 or t5.
- **Each row in the masked version** still sums to 1.0, but the probability mass is redistributed only over past/present tokens.

**What to look for:**
- The **strict lower triangle (including diagonal)** contains all non-zero weights in the masked version.
- The first row (t0) always has weight 1.0 on itself (no past context available).
- The last row (t5) can attend to all 6 tokens — it has the most context.

> **Why this matters:** GPT and other autoregressive models use causal masking to train on the task "predict the next token" — they never cheat by looking ahead. This is what enables them to generate text one token at a time at inference.

---

## Section 9 — Encoder-Decoder: Sequence-to-Sequence

### The Task: Sequence Reversal

We'll train a Transformer encoder-decoder to reverse sequences:
- Input: `[1, 2, 3, 4, 5]` → Output: `[5, 4, 3, 2, 1]`

This is a simple but clear test of sequence-to-sequence learning.

### Decoder Architecture

The Transformer Decoder has three sub-layers:
1. **Masked self-attention** — attends to previous output tokens (causal mask prevents future peeking)
2. **Cross-attention** — attends to the encoder's output (this is how the decoder "reads" the input)
3. **Feed-forward network** — same as encoder

```
Decoder Input (previous output tokens)
    │
 Masked Self-Attention (causal)
    │
 Cross-Attention (query from decoder, key/value from encoder)
    │
 Feed-Forward
    │
 Output Token Probabilities
```

In [ ]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        # Masked self-attention
        sa_out, _ = self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), tgt_mask)
        x = x + self.drop(sa_out)
        # Cross-attention
        ca_out, _ = self.cross_attn(self.norm2(x), enc_output, enc_output, src_mask)
        x = x + self.drop(ca_out)
        # FFN
        x = x + self.drop(self.ff(self.norm3(x)))
        return x


class SeqReversalTransformer(nn.Module):
    """Encoder-decoder Transformer for the reversal toy task."""
    def __init__(self, vocab_size=12, d_model=32, num_heads=2,
                 num_enc_layers=2, num_dec_layers=2, max_len=10):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.enc_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads) for _ in range(num_enc_layers)
        ])
        self.dec_layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads) for _ in range(num_dec_layers)
        ])
        self.norm_enc = nn.LayerNorm(d_model)
        self.norm_dec = nn.LayerNorm(d_model)
        self.out_proj = nn.Linear(d_model, vocab_size)

    def encode(self, src):
        pos = torch.arange(src.size(1), device=src.device).unsqueeze(0)
        x = self.embed(src) + self.pos_embed(pos)
        for layer in self.enc_layers:
            x, _ = layer(x)
        return self.norm_enc(x)

    def decode(self, tgt, enc_out):
        pos = torch.arange(tgt.size(1), device=tgt.device).unsqueeze(0)
        x = self.embed(tgt) + self.pos_embed(pos)
        tgt_len = tgt.size(1)
        causal_mask = torch.tril(torch.ones(1, 1, tgt_len, tgt_len, device=tgt.device)).bool()
        for layer in self.dec_layers:
            x = layer(x, enc_out, tgt_mask=causal_mask)
        return self.out_proj(self.norm_dec(x))

    def forward(self, src, tgt):
        enc_out = self.encode(src)
        return self.decode(tgt, enc_out)


# Generate reversal data
# Tokens: 1-10 = digits, 0 = PAD, 11 = SOS/EOS
SEQ_LEN_REV = 6
VOCAB_REV = 12  # 0=PAD, 1-10=digits, 11=SOS
SOS = 11

def gen_reversal_data(n=2000, seq_len=SEQ_LEN_REV):
    src = np.random.randint(1, 10, size=(n, seq_len))  # digits 1-9
    tgt = src[:, ::-1].copy()  # reversed
    # Decoder input: [SOS, tgt_0, tgt_1, ..., tgt_{T-1}]
    sos_col = np.full((n, 1), SOS)
    dec_in = np.hstack([sos_col, tgt[:, :-1]])  # teacher forcing input
    return src, dec_in, tgt

src_data, dec_in_data, tgt_data = gen_reversal_data(2000)

class ReversalDataset(Dataset):
    def __init__(self, src, dec_in, tgt):
        self.src = torch.tensor(src, dtype=torch.long)
        self.dec_in = torch.tensor(dec_in, dtype=torch.long)
        self.tgt = torch.tensor(tgt, dtype=torch.long)
    def __len__(self): return len(self.tgt)
    def __getitem__(self, i):
        return self.src[i], self.dec_in[i], self.tgt[i]

rev_dataset = ReversalDataset(src_data, dec_in_data, tgt_data)
rev_train, rev_val = random_split(rev_dataset, [1800, 200],
                                   generator=torch.Generator().manual_seed(42))
rev_train_loader = DataLoader(rev_train, batch_size=64, shuffle=True)

# Train
torch.manual_seed(42)
rev_model = SeqReversalTransformer()
rev_optim = optim.Adam(rev_model.parameters(), lr=1e-3)
rev_criterion = nn.CrossEntropyLoss()

from torch.utils.data import random_split

print('Training sequence reversal Transformer...')
for epoch in range(30):
    rev_model.train()
    total_loss = 0
    for src_b, dec_b, tgt_b in rev_train_loader:
        rev_optim.zero_grad()
        logits = rev_model(src_b, dec_b)  # (batch, seq, vocab)
        B, T, V = logits.shape
        loss = rev_criterion(logits.view(B*T, V), tgt_b.view(B*T))
        loss.backward()
        nn.utils.clip_grad_norm_(rev_model.parameters(), 1.0)
        rev_optim.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f'  Epoch {epoch+1}/30 | Loss: {total_loss/len(rev_train_loader):.4f}')

# Test on examples
print('\nTest Examples:')
rev_model.eval()
test_src = torch.tensor([[1,2,3,4,5,6], [7,3,9,1,4,2], [5,5,3,8,1,6]], dtype=torch.long)
with torch.no_grad():
    enc_out = rev_model.encode(test_src)
    dec_in_test = torch.full((3, 1), SOS, dtype=torch.long)
    for _ in range(SEQ_LEN_REV):
        dec_out = rev_model.decode(dec_in_test, enc_out)
        next_tok = dec_out[:, -1, :].argmax(dim=-1, keepdim=True)
        dec_in_test = torch.cat([dec_in_test, next_tok], dim=1)
    predicted = dec_in_test[:, 1:].numpy()

for i in range(3):
    print(f'  Input:     {test_src[i].tolist()}')
    print(f'  Expected:  {test_src[i].flip(0).tolist()}')
    print(f'  Predicted: {predicted[i].tolist()}')
    print()

---

## Section 10 — BERT vs GPT vs T5: A Comparison

All three are Transformer-based, but trained differently for different purposes:

| Model | Type | Pre-training Objective | Architecture | Best For |
|-------|------|------------------------|--------------|----------|
| **BERT** | Encoder-only | Masked Language Modeling (MLM) + Next Sentence Prediction | Bidirectional | Classification, NER, QA (extractive), Embeddings |
| **GPT** | Decoder-only | Causal Language Modeling (next token prediction) | Autoregressive | Text generation, completion, few-shot prompting |
| **T5** | Encoder-Decoder | "Text-to-Text" (everything framed as seq2seq) | Full encoder + decoder | Translation, summarization, QA (generative) |
| **RoBERTa** | Encoder-only | MLM (no NSP; more data, longer training) | Same as BERT | Better BERT; same tasks |
| **DistilBERT** | Encoder-only | Knowledge distillation from BERT | 6 layers (vs BERT's 12) | Fast inference; 40% smaller, 97% BERT accuracy |

### Key Insight: Bidirectional vs Autoregressive

- **BERT reads the full sequence bidirectionally** — it sees all tokens when encoding each token. Great for understanding tasks (classification, tagging).
- **GPT reads left-to-right (causal)** — it can only see past tokens. Required for generation tasks.
- **T5 uses both** — the encoder sees the full input bidirectionally; the decoder generates output autoregressively.

---

---

## Section 11 — DistilBERT Tokenization

### What Is WordPiece Tokenization?

BERT uses **WordPiece** tokenization, which splits words into sub-word units:
- "playing" → ["play", "##ing"]
- "unhappiness" → ["un", "##happi", "##ness"]

The `##` prefix indicates a continuation of a previous subword.

**Special tokens:**
- `[CLS]` — added at the start of every input; its final hidden state is used for classification
- `[SEP]` — separates two sentences (for next-sentence tasks) or marks the end

**Why subwords?**
- Handles **unknown words** — the tokenizer can always break novel words into known subwords
- Fixed vocabulary of ~30,000 tokens covers virtually any English text
- Balances between character-level (too granular) and word-level (too many unique tokens)

In [ ]:
# DistilBERT tokenization demo
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

sample_sentences = [
    "The hockey team scored three goals in the third period.",
    "Patients treated with the new medication showed significant improvement.",
    "Unhappiness and disillusionment were widespread among the players.",
    "The goalkeeper's reflexes were extraordinary during the shootout.",
    "Clinical trials demonstrated efficacy in 87% of participants."
]

print('=== DistilBERT Tokenization Examples ===\n')
for i, sent in enumerate(sample_sentences):
    enc = tokenizer(sent, truncation=True, max_length=32)
    input_ids = enc['input_ids']
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    attn_mask = enc['attention_mask']
    print(f'Sentence {i+1}: "{sent[:60]}..."')
    print(f'  Input IDs:      {input_ids}')
    print(f'  Tokens:         {tokens}')
    print(f'  Attention mask: {attn_mask}')
    subwords = [t for t in tokens if t.startswith('##')]
    print(f'  Subword tokens: {subwords if subwords else "(none)"}')
    print()

### How to Read This Output: BERT Tokenization

Each example above shows three things:

- **Input IDs:** The integer token IDs. ID 101 = `[CLS]` (always first), ID 102 = `[SEP]` (always last). Other IDs are vocabulary indices.
- **Tokens:** The human-readable WordPiece tokens. `##` prefix = this piece continues the previous word.
- **Attention mask:** 1 = real token (model should attend to it), 0 = padding (model should ignore). Since we haven't padded here, all values are 1.

**What to look for:**
- **[CLS] always appears first** — this token's final embedding is used for sequence classification.
- **[SEP] always appears last** — marks the end of the sequence.
- **Words with `##` prefix** are subwords — "unhappiness" becomes ["un", "##happi", "##ness"].
- Simple, common words like "the", "in", "a" remain as single tokens.
- Rare or complex words are split into multiple subwords.

---

## Section 12 — Fine-Tuning DistilBERT

### Why Fine-Tune Instead of Training from Scratch?

DistilBERT was pre-trained on the **entire English Wikipedia + BookCorpus** (~3.3B words). It has already learned:
- Grammar and syntax
- World knowledge
- Word and sentence semantics
- Contextual word representations

Fine-tuning **adapts** this knowledge to our specific task (hockey vs sci.med) with just:
- ~500 training examples
- 2 training epochs
- A simple linear classification head on top of [CLS]

Compare this to our from-scratch Transformer which needed many epochs and still achieved lower accuracy.

In [ ]:
# Fine-tune DistilBERT on 500 training examples for 2 epochs
# Load data
categories = ['rec.sport.hockey', 'sci.med']
newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)
texts_all = newsgroups.data
labels_all = newsgroups.target

X_tr, X_te, y_tr, y_te = train_test_split(
    texts_all, labels_all, test_size=0.3, random_state=42, stratify=labels_all
)

# Use only 500 training samples
X_tr_small = X_tr[:500]
y_tr_small = y_tr[:500]
X_te_small = X_te[:200]
y_te_small = y_te[:200]

class HFDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(list(texts), truncation=True,
                                    padding=True, max_length=max_length)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_hf = HFDataset(X_tr_small, y_tr_small, tokenizer)
eval_hf = HFDataset(X_te_small, y_te_small, tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, predictions)}

bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

training_args = TrainingArguments(
    output_dir='./distilbert_output',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='no',
    seed=42,
    no_cuda=True,  # Use CPU for reproducibility
    report_to='none',
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=eval_hf,
    compute_metrics=compute_metrics,
)

print('Fine-tuning DistilBERT (2 epochs, 500 training samples)...')
trainer.train()

results_bert = trainer.evaluate()
bert_acc = results_bert['eval_accuracy']
print(f'\nDistilBERT Fine-Tuned Accuracy (200 test examples): {bert_acc:.4f}')
print(f'Scratch Transformer Accuracy:                        {scratch_transformer_acc:.4f}')
print(f'\nAccuracy gain from pre-training: +{(bert_acc - scratch_transformer_acc)*100:.1f}%')

---

## Section 13 — Visualizing DistilBERT Attention

DistilBERT has 6 layers and 12 attention heads per layer = 72 total attention patterns. Let's extract Layer 3 attention weights for 2 examples and plot them.

In [ ]:
# Extract DistilBERT attention weights
bert_model.eval()

sample_texts_bert = [
    "The hockey team scored three goals in overtime.",
    "The patient was administered a new antiviral treatment."
]

for example_idx, text in enumerate(sample_texts_bert):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=20)
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    n_toks = len(tokens)

    with torch.no_grad():
        outputs = bert_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            output_attentions=True
        )

    # outputs.attentions: tuple of 6 tensors, each (1, 12, seq, seq)
    # Use layer 3 (0-indexed = 2), show 4 heads for space
    layer3_attn = outputs.attentions[2][0]  # (12, seq, seq)
    n_heads_show = 4

    fig, axes = plt.subplots(1, n_heads_show, figsize=(16, 4))
    for h in range(n_heads_show):
        w = layer3_attn[h, :n_toks, :n_toks].cpu().numpy()
        sns.heatmap(w, ax=axes[h], cmap='Blues', vmin=0, vmax=w.max(),
                    xticklabels=tokens, yticklabels=tokens, cbar=False)
        axes[h].set_title(f'Head {h+1}', fontsize=9)
        axes[h].tick_params(axis='x', rotation=45, labelsize=8)
        axes[h].tick_params(axis='y', rotation=0, labelsize=8)

    plt.suptitle(f'DistilBERT Layer 3 Attention — Example {example_idx+1}:\n"{text}"',
                 fontsize=11, y=1.05)
    plt.tight_layout()
    plt.show()

### How to Read This Chart: DistilBERT Attention Heads

Two rows of charts (one per example sentence), each showing 4 attention heads from DistilBERT's Layer 3.

- **Rows and columns** are labeled with the actual WordPiece tokens (including `[CLS]`, `[SEP]`, and `##` subwords).
- **Each cell** shows the attention weight from the row token to the column token.
- **Darker blue** = stronger attention.

**What to look for:**
- **[CLS] attention patterns**: `[CLS]` often attends broadly — it aggregates information from the whole sentence for classification.
- **Diagonal patterns**: A head that strongly attends to itself may be acting as a "no-op" — passing information through without transformation.
- **Domain-specific attention**: In the hockey example, does "hockey" or "goals" attract attention from other tokens? In the medical example, does "treatment" receive high cross-token attention?
- **Head 1 vs Head 4**: Different heads in the same layer can show very different patterns — this is the multi-head diversity at work.

> **Research finding:** A 2019 study found that DistilBERT's attention heads often specialize: some track syntactic dependencies, others track semantic co-reference, and some primarily attend to `[CLS]` or `[SEP]` as global information buses.

---

## Summary and Decision Guide

### The Transformer Toolkit — When to Use What

**Build from scratch when:**
- You're learning (this notebook!)
- You have domain-specific data unlike anything on the internet (niche scientific, proprietary)
- You need a tiny model (< 1M parameters) for edge deployment

**Fine-tune BERT/DistilBERT when:**
- Text classification, NER, extractive QA
- You have < 10K labeled examples
- Bidirectional context matters
- Inference latency is a concern (DistilBERT is 2x faster than BERT)

**Fine-tune GPT-2/GPT-3 when:**
- Text generation, summarization, dialogue
- Few-shot or zero-shot inference
- You need controllable generation

**Use T5/BART when:**
- Translation, summarization, abstractive QA
- Any task that benefits from seq2seq framing

### Common Mistakes to Avoid

1. **Training from scratch on small datasets** — A from-scratch Transformer needs millions of examples. Always fine-tune.
2. **Using the wrong model family** — BERT for generation will fail; GPT for classification is suboptimal.
3. **Forgetting to normalize attention scores** — Without `/ sqrt(d_k)`, attention scores explode with large d_k.
4. **Forgetting positional encoding** — Without PE, the model can't distinguish "dog bites man" from "man bites dog".
5. **Using no warmup scheduler** — Transformers benefit greatly from linear warmup for the first 10% of training steps.
6. **Setting learning rate too high** — Use 1e-5 to 5e-5 for fine-tuning. Higher rates destroy pretrained weights.

### What's Next?

The Transformer architecture is the foundation of virtually all modern AI:
- **Language:** GPT-4, Claude, Gemini — all Transformer decoders scaled to billions of parameters
- **Vision:** ViT (Vision Transformer) — images as patches treated like tokens
- **Multimodal:** CLIP, Flamingo — combining vision and language Transformers
- **Generation:** Diffusion Transformers (DiT) — the architecture behind Stable Diffusion 3, Sora

---

*Next: Notebook 05 — Generative Models: VAE and DCGAN*